In [1]:
!pip install torchsummary

In [2]:
import os
import shutil
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader
from PIL import Image
import torch.nn.functional as F
from sklearn.metrics import classification_report
from torchsummary import summary


In [4]:
# === 2. Set up EfficientNet-B0 with Transfer Learning ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Use EfficientNet's recommended weights and transforms
weights = EfficientNet_B0_Weights.DEFAULT
transform = weights.transforms()

dataset = datasets.ImageFolder('aug_processed_data', transform=transform)

# Train/val split
base_dir = 'Splited_Data'
train_dataset = datasets.ImageFolder(os.path.join(base_dir, 'train'), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(base_dir, 'val'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = 2

In [6]:
# Load model and modify classifier
model_effb0 = efficientnet_b0(weights=weights)
for param in model_effb0.parameters():
    param.requires_grad = False  # freeze base

num_features = model_effb0.classifier[1].in_features
model_effb0.classifier[1] = nn.Linear(num_features, len(train_dataset.classes))  # replace final layer
model_effb0 = model_effb0.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_effb0.classifier[1].parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:05<00:00, 4.13MB/s]


In [7]:
# === 3. Training loop ===
def train_model(epochs=10):
    best_val_acc = 0.0
    for epoch in range(epochs):
        model_effb0.train()
        total_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_effb0(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        avg_loss = total_loss / len(train_loader)

        model_effb0.eval()
        val_correct, val_total, val_loss = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model_effb0(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    print(f"\n🏆 Best Val Accuracy: {best_val_acc:.2f}%")

In [9]:
# === Run training ===
train_model(epochs=10)

Epoch 1 | Train Loss: 0.2445 | Train Acc: 94.38% | Val Loss: 0.2670 | Val Acc: 87.50%
Epoch 2 | Train Loss: 0.2233 | Train Acc: 93.12% | Val Loss: 0.2514 | Val Acc: 87.50%
Epoch 3 | Train Loss: 0.2055 | Train Acc: 97.50% | Val Loss: 0.2385 | Val Acc: 87.50%
Epoch 4 | Train Loss: 0.2010 | Train Acc: 98.12% | Val Loss: 0.2426 | Val Acc: 87.50%
Epoch 5 | Train Loss: 0.1892 | Train Acc: 97.50% | Val Loss: 0.2425 | Val Acc: 87.50%
Epoch 6 | Train Loss: 0.1654 | Train Acc: 98.12% | Val Loss: 0.2319 | Val Acc: 87.50%
Epoch 7 | Train Loss: 0.1836 | Train Acc: 96.25% | Val Loss: 0.2364 | Val Acc: 87.50%
Epoch 8 | Train Loss: 0.1817 | Train Acc: 98.12% | Val Loss: 0.2297 | Val Acc: 90.00%
Epoch 9 | Train Loss: 0.1883 | Train Acc: 96.25% | Val Loss: 0.2088 | Val Acc: 90.00%
Epoch 10 | Train Loss: 0.1568 | Train Acc: 98.12% | Val Loss: 0.2034 | Val Acc: 92.50%

🏆 Best Val Accuracy: 92.50%


Model Test

In [10]:
def predict_single_image(image_path, model, class_names):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)  # Add batch dimension

    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output, dim=1)
        _, predicted = torch.max(probs, 1)

    print(f"Predicted Class: {class_names[predicted.item()]}")
    print(f"Class Probabilities: {probs.squeeze().numpy()}")

In [11]:
# Assuming dataset = ImageFolder(...)
class_names = dataset.classes  # ['healthy', 'infected']

# Path to one test image
test_image_path_1 = "processed_data/serie infected leaves/infected_05.png"

predict_single_image(test_image_path_1, model_effb0, class_names)

Predicted Class: series_infected_leaves
Class Probabilities: [0.1325646 0.8674354]


In [12]:
test_image_path_2 = "processed_data/serie healthy leaves/healthy_05.png"
predict_single_image(test_image_path_2, model_effb0, class_names)

Predicted Class: serie_healthy_leaves
Class Probabilities: [0.6869969  0.31300312]


Model Evaluation

In [15]:
from sklearn.metrics import classification_report

def evaluate_final_model():
    model_effb0.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_effb0(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n📊 Final Evaluation on Validation Set:")
    print(classification_report(all_labels, all_preds, target_names=val_dataset.classes, digits=2))

# Run this after training
print("Evaluation Of EfficientNetB0")
evaluate_final_model()

Evaluation Of EfficientNetB0

📊 Final Evaluation on Validation Set:
                        precision    recall  f1-score   support

  serie_healthy_leaves       1.00      0.85      0.92        20
series_infected_leaves       0.87      1.00      0.93        20

              accuracy                           0.93        40
             macro avg       0.93      0.93      0.92        40
          weighted avg       0.93      0.93      0.92        40



Model Summary

In [14]:
summary(model_effb0, input_size=(3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
              SiLU-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
              SiLU-6         [-1, 32, 112, 112]               0
 AdaptiveAvgPool2d-7             [-1, 32, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             264
              SiLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 32, 1, 1]             288
          Sigmoid-11             [-1, 32, 1, 1]               0
SqueezeExcitation-12         [-1, 32, 112, 112]               0
           Conv2d-13         [-1, 16, 112, 112]             512
      BatchNorm2d-14         [-1, 16, 1